In [ ]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
from PIL import Image
import requests
import torch


save_dir = "models/gema-4b-it"

# Load model from local
model = Gemma3ForConditionalGeneration.from_pretrained(
    save_dir, device_map="auto"
).eval()

# Load processor from local
processor = AutoProcessor.from_pretrained(save_dir)

Loading checkpoint shards: 100%|██████████| 4/4 [01:20<00:00, 20.13s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
import os
import json
import re
from PIL import Image
import torch

# Đường dẫn thư mục chứa ảnh
image_dir = "VESSELimg/train"
output_json = "descriptions/VesselImg_train.json"

# Tạo dict để lưu kết quả
descriptions = {}

# Lặp qua tất cả ảnh .jpg trong thư mục
for filename in os.listdir(image_dir):
    if filename.lower().endswith(".jpg"):
        image_path = os.path.join(image_dir, filename)
        image = Image.open(image_path).convert("RGB")

        # Tạo messages như trước
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": "You are a helpful assistant."}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": "Describe the UAV-captured image over the sea, focusing on visible objects (e.g., vessels, structures, land, port, island) and environmental context (e.g., weather, time of capture, lighting, sea conditions, distance to shore), as well as the camera viewpoint (e.g., angle, altitude, orientation). Object descriptions must specify vessel type (one of six: container, passenger/ro-ro, chemical, tugboat, pilot, buoy). Limit to 40 words."}
                ]
            }
        ]

        # Tiền xử lý đầu vào
        inputs = processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt"
        ).to(model.device, dtype=torch.bfloat16)

        input_len = inputs["input_ids"].shape[-1]

        # Sinh mô tả
        with torch.inference_mode():
            generation = model.generate(**inputs, max_new_tokens=100, do_sample=False)
            generation = generation[0][input_len:]

        # Giải mã và xử lý
        decoded = processor.decode(generation, skip_special_tokens=True)
        decoded = re.sub(r"^Here['’]s a description.*?:\s*", "", decoded).strip()
        decoded = decoded.replace('"', '')

        # Ghi vào dict
        descriptions[filename] = decoded
        print(f"✓ Processed {filename}")

# Ghi toàn bộ kết quả vào file JSON
with open(output_json, "w") as f:
    json.dump(descriptions, f, indent=2)

print(f"\n✅ All descriptions saved to {output_json}")

✓ Processed group1_img_cameras_2023-10-13-13-56-57_217_jpg.rf.77e9512059147a0568516f4f6b9794ef.jpg
✓ Processed group1_img_cameras_2023-06-27-14-31-00_816_jpg.rf.722749b3c14aa1b42c25388ac6fe1ecc.jpg
✓ Processed group1_img_cameras_2023-06-27-14-04-46_155_jpg.rf.e1ef827c9bae1d343808799a458fc0f9.jpg
✓ Processed group1_img_cameras_2023-10-13-13-56-57_169_jpg.rf.f2449cd166aeb61bdd49a8b742f2cdda.jpg
✓ Processed group1_img_cameras_2023-06-27-14-05-05_211_jpg.rf.555cd5ee4e0e5633ae3a18b6205dc981.jpg
✓ Processed group1_img_cameras_2023-10-13-13-55-15_274_jpg.rf.ae553ea9acd3f830dcb65dfd777c9e11.jpg
✓ Processed group1_img_cameras_2023-10-13-13-56-57_166_jpg.rf.e17f337476ba24dc4145273636d7cb84.jpg
✓ Processed group1_img_cameras_2023-06-27-14-31-00_793_jpg.rf.9d7ed8b4d67b1fad090ce02f6da628fe.jpg
✓ Processed group1_img_cameras_2023-06-27-14-04-46_152_jpg.rf.e13e282a1b7b86a18390f3bb50d35eb5.jpg
✓ Processed group1_img_cameras_2023-10-13-13-56-57_249_jpg.rf.7cc3145e033e04c3cc5ec6fbd0596c0a.jpg
✓ Processe